# CodeSwitch-TTS — big-run notebook (2000-line re-run, ~6.5-7.5 h on T4 x2)

Standalone run sheet for the 2026-10-19 big run: the same recipe as the repository notebook
(`CodeSwitch_TTS_Kaggle.ipynb`), with the demo cells stripped. It includes the audit fixes —
residual tail windows, a `--min-epochs` training floor, corrected paper statistics — and
whisper-large-v3 as the ASR judge.

**Run order** (run cells top-to-bottom; nothing needs editing):

1. repo sync + dependencies — the kernel may restart once on a fresh image; re-run the same
   cell and continue
2. HF login (optional; reads the `HF_TOKEN` secret when it exists)
3. corpus — 2000 lines, register-balanced
4. teacher pass — ~4.5 h → **Save Version** (so `teacher_cache/` survives a session death)
5. train the student — ~30-50 min; `--min-epochs 300` uses the whole wall-clock budget
6. CB0-only diagnostic — listen: is the draft intelligible with the residual muted?
7. full benchmark — writes `eval_results_2k/`, `paper_figures_2k/`, `eval_audio_2k/`
8. push to W&B (optional; needs the `WANDB_API_KEY` secret + Internet) — a safety net if
   the session dies before you download
9. zip + download — send the zip back for analysis

Do **not** skip step 6: if the residual-muted audio is also noise, stop and send the wavs
before spending eval time.


## 1. Repository sync + dependencies (~2 min; may restart the kernel once)


In [ ]:
import os, sys, importlib.metadata as md

REPO_URL = "https://github.com/tripathiji1312/codeSwitch.git"
CLONE_DIR = "/kaggle/working/codeSwitch"

# --- 1. Always sync the repository (fast-forward the clone, or clone fresh) ---
if os.path.exists(os.path.join(CLONE_DIR, ".git")):
    if os.system(f"git -C {CLONE_DIR} fetch origin main") != 0:
        os.system(f"rm -rf {CLONE_DIR}")
        os.system(f"git clone {REPO_URL} {CLONE_DIR}")
    else:
        os.system(f"git -C {CLONE_DIR} reset --hard FETCH_HEAD")
else:
    os.system(f"rm -rf {CLONE_DIR}")
    os.system(f"git clone {REPO_URL} {CLONE_DIR}")

# --- 2. Decide from the installed state (not a marker file) whether to (re)install ---
def _deps_ok() -> bool:
    try:
        if tuple(map(int, md.version("protobuf").split(".")[:2])) < (5, 26):
            return False                        # no google.protobuf.runtime_version
        if md.version("transformers") != "4.46.1":
            return False
        md.version("parler-tts")                # raises if parler-tts is missing
        md.version("sentencepiece")
        md.version("jiwer")                     # evaluation suite (CER/WER)
        md.version("matplotlib")                # paper figures
        return True
    except Exception:
        return False

if not _deps_ok():
    print("Installing dependencies (staged to avoid package conflicts)...")
    # Stage 1: parler-tts from GitHub (brings its transformers/audiotools chain)
    os.system("pip install -q -U git+https://github.com/huggingface/parler-tts.git")
    # Stage 2: pin transformers to the exact version parler-tts v0.2.2 targets
    os.system("pip install -q 'transformers==4.46.1'")
    # Stage 3: remaining audio/scientific libs
    os.system("pip install -q descript-audio-codec soundfile scipy accelerate sentencepiece")
    # Stage 3b: evaluation-suite extras (CER/WER + IEEE-style figures)
    os.system("pip install -q jiwer matplotlib seaborn pandas")
    # Stage 4: protobuf LAST and >= 5.26 (it ships google.protobuf.runtime_version).
    # parler-tts -> descript-audiotools pins protobuf<5 and silently downgrades it to
    # 4.25.9, which broke the Parler-TTS import and forced the noise-only fallback.
    os.system("pip install -q 'protobuf==5.29.5'")
    print("✓ Dependencies installed!")
    print("⚠ The kernel now restarts so the new protobuf/transformers take effect.")
    print("👉 Click 'Run All' again — installation is skipped on the second pass.")
    os._exit(0)  # hard restart; a stale protobuf 4.x cannot be swapped in-process
else:
    print("✓ Dependencies already installed.")

print(f"   protobuf:     {md.version('protobuf')}  (needs >= 5.26)")
print(f"   transformers: {md.version('transformers')}")

# --- 3. Sanity-check the Parler-TTS import (the real cause of the static-noise runs) ---
try:
    from parler_tts import ParlerTTSForConditionalGeneration  # noqa: F401
    print("✓ parler_tts import OK")
except Exception as e:
    print(f"⚠ parler_tts import FAILED: {type(e).__name__}: {e}")
    print("  Fix: !pip install -q 'protobuf==5.29.5'  then restart the session (Stop & Run All).")

# --- 4. Make the packaged code importable ---
for path_candidate in [CLONE_DIR, ".", ".."]:
    if os.path.exists(os.path.join(path_candidate, "codeswitch_tts")):
        if path_candidate not in sys.path:
            sys.path.insert(0, path_candidate)
        print(f"✓ Root path configured: {path_candidate}")
        break


If the cell above restarted the kernel (a fresh image without the pinned dependencies), run
it once more — the second pass is a fast no-op — then continue.


In [ ]:
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "x", torch.cuda.device_count())
!git -C /kaggle/working/codeSwitch log --oneline -3


## 2. HF login

Reads `HF_TOKEN` from Kaggle **Add-ons → Secrets**. The cell degrades gracefully when the
secret is not set — the models used here are public.


In [ ]:
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
        print("✓ Hugging Face login successful via Kaggle Secrets!")
    else:
        print("ℹ HF_TOKEN not set in Kaggle Secrets; proceeding with anonymous downloads.")
except Exception as e:
    print(f"ℹ Proceeding with public access ({e}).")


## 3. Corpus — 2000 lines (~5.5 h of teacher time in step 4)

Register-balanced (34/33/33 native Tamil / romanised / transit), unique sentences, disjoint
from the 60-utterance benchmark. Selection is deterministic and prefix-stable: raising
`--max-sentences` later only adds sentences, and only the new lines cost teacher time.


In [ ]:
import os
os.chdir('/kaggle/working/codeSwitch')
# 2000 lines at 34/33/33 - the file is what the teacher pass pays for, so --max-sentences
# is the knob that makes this fit one T4 session (~5.5 h of teacher time).
!python build_codeswitch_corpus.py --templates 2500 --wikipedia 900 --max-sentences 2000 \
    --out codeswitch_corpus.txt --json-out codeswitch_corpus.stats.json


## 4. Teacher pass — run alone (~5.5 h), then W&B backup

Collection is cached by sentence content (`teacher_cache/shard_<sha1>.npz`), so Ctrl-C or a
session death loses nothing — re-running resumes where it stopped (it prints
`cache: N/2000 sentences already collected ...` up front). The next cell **backs the finished
cache up to W&B automatically** (`--group teacher`), which is the recovery path if the session
later dies: `python push_artifacts_to_wandb.py --download teacher --dest .`. Kaggle *Quick
Save* is not a reliable file backup — do not rely on it.


In [ ]:
import os
os.chdir('/kaggle/working/codeSwitch')
# ---- phase 1: teacher only (~5.5 h for 2000 lines; Ctrl-C / session death is safe) ----
# The run ends with an automatic W&B backup of the cache (last lines of this cell) —
# that backup is the recovery path if the session later dies. Then continue with phase 2.
# --num-sentences must match the corpus size — the trainer silently caps at 300 without it.
!python train_draft_distill.py --corpus-json codeswitch_corpus.txt \
    --num-sentences 2000 --skip-draft --skip-residual --ckpt-dir codeswitch_ckpt

# Back up the finished teacher cache to W&B (this is the ~5.5 h a dead session would eat).
# Recover in any fresh session with:  !python push_artifacts_to_wandb.py --download teacher --dest .
!pip install -q wandb
!python push_artifacts_to_wandb.py --group teacher


## 5. Train the student (~30-50 min)

`--min-epochs 300` keeps early stopping from ending training before the wall-clock caps
(`--max-minutes 30` / `--max-minutes-residual 20`) bind; the best-val checkpoint is still
restored at the end. Watch for a `[Residual] windows cover ...` line — with the tail-window
fix it should be absent or tiny (it logs the frames the 24-frame minimum drops).


In [ ]:
import os
os.chdir('/kaggle/working/codeSwitch')
# ---- phase 2: draft + residual on the finished cache (~30-50 min wall-clock) ----
# --min-epochs 300 keeps early stopping from ending training before the wall-clock caps
# bind (the best-val checkpoint is still restored), so a noisy val split cannot leave the
# student undertrained again.
# --num-sentences must match the corpus size — the trainer silently caps at 300 without it.
!python train_draft_distill.py --corpus-json codeswitch_corpus.txt \
    --num-sentences 2000 --draft-epochs 60 --residual-sched 0.5 --min-epochs 300 \
    --max-minutes 30 --max-minutes-residual 20 --ckpt-dir codeswitch_ckpt
# watch for a `[Residual] windows cover ...` line: it logs frames the 24-frame minimum
# drops; with the tail-window fix it should be absent or tiny on 2000-line shards.

# Back up the trained student (draft + residual + meta.json) to W&B as well:
#   !python push_artifacts_to_wandb.py --download student --dest .
!pip install -q wandb
!python push_artifacts_to_wandb.py --group student


## 6. CB0-only diagnostic (decision point, ~10 min)

Synthesises **one** utterance with config C and, per streaming chunk, saves two wavs:

* `chunkN_cb0only.wav` — draft + rejection sampling only (codebooks 1-8 muted): is the *text* intelligible?
* `chunkN_full.wav` — what config C actually sounds like end to end.

It also prints the new `codeswitch_ckpt/meta.json` (draft val top-1 vs chance; residual
`cb_top1`). Read the pairs before spending time on the benchmark:

* cb0-only intelligible, full = noise → the residual decoder is still the weak point.
* **both** noise → the draft itself is failing at this scale: stop and send the wavs before
  spending eval time.

The patched `decode_codes` mutes codebooks 1-8 on the same decode path the engine uses, so
the comparison isolates exactly the residual contribution.


In [ ]:
# ===== CB0-only diagnostic (Kaggle, run in the SAME session as the eval run) =====
# Answers: is the draft+rejection-sampled Codebook-0 audio intelligible, and are the
# last 8 codebooks (residual decoder) the thing that turns B/C into "mic rubbing" noise?
# Runs config C on ONE utterance with ASR/UTMOS off. ~5-8 min (model load dominates).
import os, sys, json, pathlib

for cand in ("/kaggle/working/codeSwitch", "codeSwitch", "."):
    if os.path.isfile(os.path.join(cand, "run_full_evaluation.py")):
        os.chdir(cand)
        break
else:
    raise SystemExit("could not find run_full_evaluation.py — cd to the repo first")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

import run_full_evaluation as R
from codeswitch_tts.utils.audio import save_wav

OUT = pathlib.Path("/kaggle/working/cb0diag")
OUT.mkdir(parents=True, exist_ok=True)
CKPT = "codeswitch_ckpt"

# --- Step 0: what the 300-sentence checkpoints actually learned ----------
meta_path = pathlib.Path(CKPT) / "meta.json"
if meta_path.is_file():
    print(json.dumps(json.loads(meta_path.read_text()), indent=2)[:4000])
else:
    print("no meta.json; ckpt dir:", sorted(p.name for p in pathlib.Path(CKPT).iterdir()))

# --- Step 1: capture each decoded chunk + a CB1-8-muted twin -------------
_orig_decode = R.EvalSynthesizer.decode_codes
MAX_CHUNKS = 10
_saved = 0

def _patched_decode_codes(self, codes):
    global _saved
    wave = _orig_decode(self, codes)   # unchanged: this is the audio you heard
    try:
        if _saved < MAX_CHUNKS and codes.dim() == 2 and codes.shape[0] == 9:
            idx = _saved + 1
            cb0_only = codes.clone()
            cb0_only[1:] = 0                     # mute codebooks 1..8 (constant token 0)
            wave_cb0 = _orig_decode(self, cb0_only)   # SAME decode path as above
            save_wav(OUT / f"chunk{idx}_full.wav", wave, 44100)
            save_wav(OUT / f"chunk{idx}_cb0only.wav", wave_cb0, 44100)
            _saved = idx
            print(
                f"[diag] chunk {idx}: {codes.shape[-1]} frames | "
                f"token range {int(codes.min())}..{int(codes.max())} | "
                f"saved -> cb0diag/chunk{idx}_full.wav + chunk{idx}_cb0only.wav"
            )
    except Exception as exc:
        print(f"[diag] capture skipped on this chunk ({type(exc).__name__}: {exc})")
    return wave

R.EvalSynthesizer.decode_codes = _patched_decode_codes

try:
    rc = R.main([
        "--configs", "C",
        "--limit", "1",                 # first utterance only
        "--gate-thresholds", "0",       # the un-gated hybrid (the noisy config)
        "--skip-asr", "--no-utmos",     # keep it fast; measurement not needed here
        "--ckpt", CKPT,
        "--results", str(OUT / "raw_results.jsonl"),
        "--audio-dir", str(OUT / "audio"),
        "--figures-dir", str(OUT / "figs"),
    ])
    print("main() ->", rc)
except Exception as exc:
    print("main() raised:", type(exc).__name__, exc)

print(f"captured {_saved} chunk pairs")
print("wavs:", sorted(p.name for p in OUT.glob("*.wav")))


In [ ]:
from IPython.display import Audio, display
import glob

pairs = sorted(glob.glob('/kaggle/working/cb0diag/chunk*_full.wav'))
print(f"{len(pairs)} chunk pair(s) in /kaggle/working/cb0diag")
for p in pairs[:4]:
    twin = p.replace('_full.wav', '_cb0only.wav')
    print('FULL     - what config C sounds like:', p.split('/')[-1])
    display(Audio(p))
    print('CB0-ONLY - residual muted:', twin.split('/')[-1])
    display(Audio(twin))


## 7. Full benchmark (~30-60 min; the first run downloads whisper-large-v3, ~3 GB)

Runs configs A (pure AR), B (pure NAR) and C (the hybrid) on all 60 benchmark utterances and
writes figures + LaTeX tables (I–V) into the fresh `paper_figures_2k/`. Results append after
every utterance — re-run the cell to resume. The provenance guard refuses to mix rows from
different checkpoints (exit 3; `--force` re-synthesises). Keep the default derived gate sweep;
`--gate-mode synth` is only for an end-to-end gated mixture.


In [ ]:
import os
os.chdir('/kaggle/working/codeSwitch')
!python run_full_evaluation.py --ckpt codeswitch_ckpt \
    --asr-model openai/whisper-large-v3 \
    --results eval_results_2k/raw_results.jsonl \
    --audio-dir eval_audio_2k --figures-dir paper_figures_2k


## 8. Push to Weights & Biases (optional)

Uploads scalars, figures, tables, best/median/worst-CER audio per config, and the raw
results as a W&B artifact — the safety net if the session dies before you download the
zip.

Needs two things: the `WANDB_API_KEY` Kaggle secret (Add-ons → Secrets) and the
notebook's *Internet* switch on. Skip this cell otherwise — it errors harmlessly and you
lose nothing.


In [ ]:
import os
os.chdir('/kaggle/working/codeSwitch')
!pip install -q wandb
# WANDB_API_KEY comes from the Kaggle secret of the same name.
!python push_to_wandb.py --project codeswitch-tts --tags kaggle,t4 \
    --figures-dir paper_figures_2k --audio-dir eval_audio_2k --ckpt-dir codeswitch_ckpt \
    --results eval_results_2k/raw_results.jsonl --max-audio 4


## 9. Zip the artifacts


In [ ]:
import os
os.chdir('/kaggle/working/codeSwitch')
# Figures + tables + metrics + audio samples + CB0-only diagnostic wavs (weights excluded)
!zip -qr /kaggle/working/codeswitch_artifacts_2k.zip paper_figures_2k eval_results_2k eval_audio_2k codeswitch_ckpt cb0diag -x 'codeswitch_ckpt/*.pt'
!ls -lh /kaggle/working/codeswitch_artifacts_2k.zip

# Back the final zip up to W&B too (last safety net):
#   !python push_artifacts_to_wandb.py --download bundle --dest /kaggle/working
!pip install -q wandb
!python push_artifacts_to_wandb.py --group bundle --zip /kaggle/working/codeswitch_artifacts_2k.zip


## Send back

`/kaggle/working/codeswitch_artifacts_2k.zip` — or at minimum:

* `codeswitch_ckpt/meta.json`
* `eval_results_2k/raw_results.jsonl` + `eval_results_2k/run_meta.json`
* `paper_figures_2k/tables.tex` + `paper_figures_2k/summary.json`
* 2-3 `cb0diag/` `chunkN_full.wav` / `chunkN_cb0only.wav` pairs

Notes: a dead session loses nothing — the teacher cache, the student checkpoints and the final
zip are auto-backed-up to W&B after each step (`python push_artifacts_to_wandb.py --download
teacher --dest .`, `student`, `bundle` to restore), and eval rows are resumable. The backups
(and the optional dashboard in step 8) need a `WANDB_API_KEY` secret; without it they warn and
continue.
